# K-means sobre el dataset ampliado de Shapes

En este notebook se utiliza el dataset de 100 elementos generado mediante oversampling sintético.  
El objetivo es:

1. Cargar el CSV desde GitHub.
2. Preparar las variables categóricas para aplicar K-means.
3. Calcular una propuesta de `k` óptima mediante el método del codo y la silueta.
4. Aplicar el algoritmo K-means.
5. Analizar los clusters resultantes.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

try:
    from sklearn.preprocessing import OneHotEncoder
    encoder_kwargs = {"sparse_output": False, "handle_unknown": "ignore"}
except TypeError:
    from sklearn.preprocessing import OneHotEncoder
    encoder_kwargs = {"sparse": False, "handle_unknown": "ignore"}

RANDOM_STATE = 26

## 1. Carga del dataset

El dataset se carga directamente desde el archivo CSV publicado en GitHub.  
Como el archivo está delimitado por punto y coma (`;`), se indica `sep=";"` en `read_csv`.

In [6]:
from pandas.api.types import is_object_dtype, is_string_dtype

# URL del dataset en formato raw
url = "https://raw.githubusercontent.com/danielagarciachavero/AC/refs/heads/main/PR/DanielaGarciaChavero_EJ1_3.csv"

# Carreguem el CSV utilitzant el separador correcte
df = pd.read_csv(url, sep=";")

# Netegem possibles espais en els noms de columnes i valors de text
df.columns = df.columns.str.strip()

for col in df.columns:
    if is_object_dtype(df[col]) or is_string_dtype(df[col]):
        df[col] = df[col].astype("string").str.strip()

print("Dimensiones del dataset:", df.shape)
display(df.head(16))

Dimensiones del dataset: (100, 5)


,Forma,Numero_lados,Color,Tamano,Direccion
0,Triangulo,3,Verde,Pequeno,Arriba
1,Triangulo,3,Azul,Pequeno,Derecha
2,Triangulo,3,Rojo,Grande,Derecha
3,Triangulo,3,Azul,Grande,Arriba
4,Triangulo,3,Azul,Grande,Derecha
5,Triangulo,3,Amarillo,Grande,Derecha
6,Triangulo,3,Rojo,Grande,Arriba
7,Triangulo,3,Amarillo,Grande,Arriba
8,Triangulo,3,Verde,Grande,Arriba
9,Triangulo,3,Rojo,Grande,Arriba


## 2. Revisión básica del dataset

Se comprueba que el dataset tenga 100 registros y que las columnas coincidan con los atributos definidos: `Forma`, `Numero_lados`, `Color`, `Tamano` y `Direccion`.

In [ ]:
columnas_esperadas = ["Forma", "Numero_lados", "Color", "Tamano", "Direccion"]

# Validem que hi siguin totes les columnes esperades
columnas_faltantes = [col for col in columnas_esperadas if col not in df.columns]

if columnas_faltantes:
    raise ValueError(f"Faltan columnas en el dataset: {columnas_faltantes}")

df = df[columnas_esperadas].copy()

print("Número de registros:", len(df))
print("\nValores nulos por columna:")
display(df.isna().sum())

print("\nDistribución de formas:")
display(df["Forma"].value_counts())

print("\nDistribución de colores:")
display(df["Color"].value_counts())

print("\nDistribución de tamaños:")
display(df["Tamano"].value_counts())

print("\nDistribución de direcciones:")
display(df["Direccion"].value_counts())

## 3. Preparación de los datos para K-means

K-means trabaja con variables numéricas. Como este dataset contiene variables categóricas, se utiliza **one-hot encoding** para convertir cada categoría en una columna binaria.

Aunque `Numero_lados` es numérica, aquí se trata como variable categórica porque depende directamente de la forma y no representa una escala continua.

In [ ]:
# Tractem totes les variables com a categòriques per evitar relacions ordinals artificials
X_categorico = df.astype(str)

encoder = OneHotEncoder(**encoder_kwargs)
X = encoder.fit_transform(X_categorico)

feature_names = encoder.get_feature_names_out(columnas_esperadas)
X_encoded = pd.DataFrame(X, columns=feature_names)

print("Dimensiones después del one-hot encoding:", X_encoded.shape)
display(X_encoded.head())

## 4. Cálculo de la k óptima

Se calculan dos métricas para distintos valores de `k`:

- **Inercia**: se usa para el método del codo. Valores más bajos indican clusters más compactos.
- **Coeficiente de silueta**: mide la separación entre clusters. Valores más altos indican una mejor estructura de agrupación.

En este notebook se toma como `k` principal el valor que maximiza la silueta, y se muestra también la sugerencia del método del codo.

In [ ]:
def detectar_codo(k_values, inercias):
    """Detecta el codo como el punto con mayor distancia a la línea entre extremos."""
    puntos = np.column_stack((k_values, inercias))

    primero = puntos[0]
    ultimo = puntos[-1]

    linea = ultimo - primero
    norma_linea = np.linalg.norm(linea)

    if norma_linea == 0:
        return k_values[0]

    distancias = []
    for punto in puntos:
        distancia = np.abs(np.cross(linea, primero - punto)) / norma_linea
        distancias.append(distancia)

    return int(k_values[np.argmax(distancias)])


max_k = min(10, len(df) - 1)
k_values = list(range(1, max_k + 1))

inercias = []
siluetas = []

for k in k_values:
    modelo = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    etiquetas = modelo.fit_predict(X_encoded)
    inercias.append(modelo.inertia_)

    if k >= 2:
        siluetas.append(silhouette_score(X_encoded, etiquetas))

k_values_silueta = k_values[1:]

k_codo = detectar_codo(np.array(k_values), np.array(inercias))
k_silueta = int(k_values_silueta[np.argmax(siluetas)])

print("k sugerida por el método del codo:", k_codo)
print("k sugerida por el coeficiente de silueta:", k_silueta)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(k_values, inercias, marker="o")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inercia")
plt.title("Método del codo")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(k_values_silueta, siluetas, marker="o")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Coeficiente de silueta")
plt.title("Análisis de silueta")
plt.grid(True)
plt.show()

## 5. Aplicación de K-means

Se aplica K-means utilizando la `k` seleccionada a partir del coeficiente de silueta.  
Si se quiere forzar otro valor, se puede modificar manualmente la variable `k_optima`.

In [ ]:
k_optima = k_silueta

modelo_kmeans = KMeans(
    n_clusters=k_optima,
    random_state=RANDOM_STATE,
    n_init=20
)

df_clusterizado = df.copy()
df_clusterizado["Cluster"] = modelo_kmeans.fit_predict(X_encoded)

print("k utilizada:", k_optima)
display(df_clusterizado.head(20))

## 6. Análisis de los clusters resultantes

En esta sección se analiza el tamaño de cada cluster y las características predominantes dentro de cada grupo.

In [ ]:
print("Tamaño de cada cluster:")
display(df_clusterizado["Cluster"].value_counts().sort_index())

# Moda de cada atributo per cluster
resumen_clusters = (
    df_clusterizado
    .groupby("Cluster")[columnas_esperadas]
    .agg(lambda x: x.mode().iloc[0])
)

resumen_clusters["Num_elementos"] = df_clusterizado.groupby("Cluster").size()

print("Características dominantes por cluster:")
display(resumen_clusters)

In [ ]:
# Taules creuades per entendre millor cada cluster
for atributo in ["Forma", "Color", "Tamano", "Direccion"]:
    print(f"Distribución de {atributo} por cluster:")
    tabla = pd.crosstab(df_clusterizado["Cluster"], df_clusterizado[atributo])
    display(tabla)

## 7. Visualización de los clusters

Para visualizar los resultados se reduce el dataset codificado a dos dimensiones mediante PCA.  
Esta visualización no se usa para entrenar el modelo, solo para representar gráficamente los clusters.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_encoded)

df_visual = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Cluster": df_clusterizado["Cluster"]
})

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    df_visual["PC1"],
    df_visual["PC2"],
    c=df_visual["Cluster"],
    alpha=0.8
)
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Visualización de clusters con PCA")
plt.grid(True)
plt.colorbar(scatter, label="Cluster")
plt.show()

print("Varianza explicada por PC1 y PC2:")
display(pd.Series(pca.explained_variance_ratio_, index=["PC1", "PC2"]))

## 9. Interpretación final

Para redactar el análisis final, se recomienda comentar:

- Qué valor de `k` se ha considerado óptimo según la silueta y el método del codo.
- Si los clusters se explican principalmente por la forma, el color, el tamaño o la dirección.
- Si los grupos obtenidos son coherentes con la estructura del dataset original.
- Que el dataset contiene variables categóricas transformadas mediante one-hot encoding para poder aplicar K-means.